# Contextual graph-kernel similarity

This benchmark evaluates higher-order similarity on frozen BridgeRNA layer-12 contextual graphs. It tests graph similarity, not GRN inference. RR1/RR3 did not determine any graph or kernel parameter.

In [1]:
from pathlib import Path
import json, pandas as pd
from IPython.display import display
R=Path('results')
decision=json.loads((R/'summary/decision.json').read_text())
display(pd.Series(decision).to_frame('value'))

,value
decision,C. SIMPLE GRAPH METRICS ARE SUFFICIENT
best_kernel,Spectral
best_subset_tissue_MRR,0.334732
primary_perturbation_endpoint,unavailable without defining an unvalidated si...
propagation_kernel,unavailable
tissue_subset_samples,140
claim_limitation,"graph similarity, not GRN inference"


## Methods and tractability

WL uses two subtree-refinement iterations with identity-aware and structural labels. Shortest-path uses eight fixed landmarks, graphlet uses 1,000 deterministic 3-node and 4-node samples, and spectral similarity uses 16 leading normalized weighted-adjacency eigenvalues. Higher-order tissue kernels use a prespecified study-diverse subset of 140 samples.

In [2]:
display(pd.read_csv(R/'tissue/subset_manifest.csv').groupby('tissue').agg(samples=('gsm','size'),studies=('gse','nunique')).reset_index())
display(pd.read_csv(R/'summary/kernel_summary.csv').style.format({'R@1':'{:.1%}','R@5':'{:.1%}','R@10':'{:.1%}','MRR':'{:.3f}','runtime_seconds':'{:.1f}'}).hide(axis='index'))

,tissue,samples,studies
0,adipose,10,10
1,blood,10,10
2,brain,10,10
3,breast,10,10
4,colon,10,10
5,heart,10,10
6,kidney,10,10
7,liver,10,10
8,lung,10,10
9,ovary,10,10


endpoint,method,queries,R@1,R@5,R@10,MRR,median_rank,runtime_seconds,peak_RAM_MB
technical,WL identity-aware,80,0.0%,0.0%,0.0%,0.019,53.500000,26.3,811.773438
technical,WL structural,80,2.5%,5.0%,6.2%,0.064,45.000000,24.7,811.773438
technical,Shortest-path,80,1.2%,2.5%,2.5%,0.035,58.000000,293.3,811.773438
technical,Graphlet,80,2.5%,5.0%,6.2%,0.058,55.000000,26.4,811.773438
technical,Spectral,80,1.2%,11.2%,21.2%,0.084,26.500000,45.8,811.773438
tissue,WL identity-aware,140,10.0%,30.0%,50.7%,0.231,10.000000,43.8,1466.925781
tissue,WL structural,140,14.3%,40.7%,59.3%,0.268,8.500000,40.8,1466.925781
tissue,Shortest-path,140,8.6%,35.7%,52.1%,0.226,9.000000,511.5,1466.925781
tissue,Graphlet,140,6.4%,30.7%,55.7%,0.197,10.000000,43.8,1466.925781
tissue,Spectral,140,20.0%,47.1%,66.4%,0.335,6.000000,82.9,1466.925781


## Technical-pair retrieval

Forty controlled same-RNA T-cell PolyA/Ribo pairs provide the independent technical-retrieval sanity check.

In [3]:
display(pd.read_csv(R/'technical/kernel_retrieval.csv').style.format({'R@1':'{:.1%}','R@5':'{:.1%}','R@10':'{:.1%}','MRR':'{:.3f}'}).hide(axis='index'))

endpoint,method,queries,R@1,R@5,R@10,MRR,median_rank,runtime_seconds,peak_RAM_MB
technical,WL identity-aware,80,0.0%,0.0%,0.0%,0.019,53.500000,26.303683,811.773438
technical,WL structural,80,2.5%,5.0%,6.2%,0.064,45.000000,24.658490,811.773438
technical,Shortest-path,80,1.2%,2.5%,2.5%,0.035,58.000000,293.280598,811.773438
technical,Graphlet,80,2.5%,5.0%,6.2%,0.058,55.000000,26.404365,811.773438
technical,Spectral,80,1.2%,11.2%,21.2%,0.084,26.500000,45.828210,811.773438


## Study-disjoint tissue retrieval

Candidates from the query GSE are excluded. Full-cohort non-graph neighborhood purity is shown separately because it is not numerically equivalent to subset retrieval MRR.

In [4]:
display(pd.read_csv(R/'tissue/kernel_retrieval.csv').style.format({'R@1':'{:.1%}','R@5':'{:.1%}','MRR':'{:.3f}'}).hide(axis='index'))
display(pd.read_csv(R/'tissue/non_graph_full_cohort_reference.csv'))

endpoint,method,queries,R@1,R@5,R@10,MRR,median_rank,runtime_seconds,peak_RAM_MB
tissue,WL identity-aware,140,10.0%,30.0%,0.507143,0.231,10.000000,43.811741,1466.925781
tissue,WL structural,140,14.3%,40.7%,0.592857,0.268,8.500000,40.755461,1466.925781
tissue,Shortest-path,140,8.6%,35.7%,0.521429,0.226,9.000000,511.516432,1466.925781
tissue,Graphlet,140,6.4%,30.7%,0.557143,0.197,10.000000,43.793159,1466.925781
tissue,Spectral,140,20.0%,47.1%,0.664286,0.335,6.000000,82.942711,1466.925781


,representation,k,tissue_purity,study_purity,tissue_to_study_ratio
0,raw15165,5,0.868582,0.142237,6.106575
1,raw15165,10,0.831571,0.075733,10.980226
2,raw15165,25,0.770856,0.032103,24.012186
3,raw15165,50,0.702726,0.016693,42.096668
4,pca15165_512,5,0.856112,0.142359,6.013740
5,pca15165_512,10,0.818490,0.075581,10.829357
6,pca15165_512,25,0.755122,0.032225,23.432853
7,pca15165_512,50,0.692867,0.016815,41.204289
8,bridgerna512,5,0.818093,0.120782,6.773279
9,bridgerna512,10,0.773686,0.066259,11.676661


## Perturbation endpoint and RR1/RR3 stress test

Standard kernels compare complete graphs. A treatment-minus-control graph feature difference is signed and is not itself a validated positive-semidefinite graph kernel. The benchmark does not invent one merely to fill the primary endpoint. Existing signed-edge response similarities are retained as secondary stress references.

In [5]:
display(pd.read_csv(R/'response/status.csv'))
display(pd.read_csv(R/'stress/simple_graph_stress.csv').query("k == 10 and graph == 'union'"))

,method,status,reason
0,higher-order response kernels,not_estimable_from_sample-kernel features with...,WL/graphlet feature differences can be signed ...


,k,graph,comparison,signed_edge_cosine,signed_edge_correlation,edges_A,edges_B
8,10,union,RR1,0.183429,0.184914,204759,209271
9,10,union,RR3-39,0.319309,0.319231,163894,163303
10,10,union,RR3-40,0.399024,0.399005,160472,161860
11,10,union,RR1↔RR3-39 false friend,0.011796,0.011390,204759,163894


## Null control

The technical feature-label permutation checks retrieval specificity. Structural-kernel invariance to node relabeling is expected; prior degree-matched random-graph results remain the relevant structural control.

In [6]:
display(pd.read_csv(R/'nulls/technical_null.csv').style.format({'observed_MRR':'{:.3f}','label_shuffled_MRR':'{:.3f}','degradation':'{:.3f}'}).hide(axis='index'))

method,observed_MRR,label_shuffled_MRR,degradation,null_note
WL identity-aware,0.019,0.044,-0.024,sample-feature permutation; structural graph randomization retained from prior validation
WL structural,0.064,0.101,-0.037,sample-feature permutation; structural graph randomization retained from prior validation
Shortest-path,0.035,0.069,-0.033,sample-feature permutation; structural graph randomization retained from prior validation
Graphlet,0.058,0.069,-0.011,sample-feature permutation; structural graph randomization retained from prior validation
Spectral,0.084,0.086,-0.002,sample-feature permutation; structural graph randomization retained from prior validation


## Final decision

The higher-order kernels do not establish a better perturbation-response retrieval method. They may serve as secondary sample-graph similarities, but their computational cost and modest tissue retrieval do not justify replacing the validated simple graph metrics.

In [7]:
display(pd.read_csv(R/'summary/important_output_table.csv').style.format(na_rep='—').hide(axis='index'))

Method,Technical MRR,Tissue MRR,Perturbation MRR,RR3-39 rank,RR3-40 rank,RR1 rank,False-friend rank,Null degradation,Runtime seconds
Raw,—,—,—,—,—,—,—,—,—
PCA,—,—,—,—,—,—,—,—,—
Bridge mean,—,—,—,—,—,—,—,—,—
Hallmark mean+SD,—,—,—,—,—,—,—,—,—
Edge Jaccard,—,—,—,—,—,—,—,—,—
Neighborhood Jaccard,—,—,—,—,—,—,—,—,—
Weighted edge,—,—,—,—,—,—,—,—,—
WL identity-aware,0.019163,0.231261,—,—,—,—,—,—,70.115424
WL structural,0.064299,0.267721,—,—,—,—,—,—,65.413950
Shortest-path,0.035422,0.226010,—,—,—,—,—,—,804.797030
